In [0]:
# =============================================================================
# recon_expected_vs_actual  -  READ ONLY. Validate the pipeline ACTUAL counts against
# BELLA's EXPECTED counts, for EVERY active state + EVERY archive segment. This is the
# "are the numbers right" check (the Overall recon proves internal consistency + no-dup;
# THIS proves they equal Bella's figures).
#   - Fill EXPECTED below from Bella (active 16 states + archive FTA/UTA/FPA/TD).
#   - Actuals are read live from stg_segmentation_states + the archive stg tables.
#   - Output: per-row expected | actual | diff | MATCH, single print + downloadable workbook.
# =============================================================================

In [0]:
# ---- CELL 0 : config + auth + EXPECTED (edit archive rows when Bella sends them) ----
from pyspark.sql import functions as F
from pyspark.sql.functions import *
import io, base64, datetime, builtins
pysum = builtins.sum   # `import *` shadows Python sum with Spark's -> use pysum for plain totals
REPORT=[]
def log(*a): REPORT.append(" ".join(str(x) for x in a))

# ACTIVE expected (Bella, via segmentation-v5). Confirm/replace with Bella's first-hand numbers.
ACTIVE_EXPECTED = {
 "appealSubmitted":43, "awaitingRespondentEvidence(a)":9, "awaitingRespondentEvidence(b)":42,
 "caseUnderReview":177, "decided(a)":2004, "decided(b)":21, "decision":331, "ended":424,
 "ftpaSubmitted(a)":301, "ftpaDecided":1255, "ftpaSubmitted(b)":124, "listing":436,
 "paymentPending":19, "prepareForHearing":1079, "reasonsForAppealSubmitted":109, "remitted":29,
}
# ARCHIVE expected per segment (Bella confirmed 2026-08-14): FTA=120485 (the changed one);
# all other segments are "SAME AS BEFORE" (unchanged by the seg change).
#  - UTA 7960 / FPA 429 = independent pre-change values from the pre2307 baseline snapshot
#    (so a difference here is a REAL detected change -> this is what flags the UTA -1).
#  - TD 1807728 = "same as before": TD only received the 2048 date param (Andrew: same count
#    before/after). The pre2307 snapshot did NOT capture TD, so this is the current confirmed
#    value; to make it an INDEPENDENT hard-check, replace with the TD count from a pre-change
#    Overall recon run (A_06_TD_stg_total from the 08:40 run, before archive was re-run).
ARCHIVE_EXPECTED = { "FTA":120485, "UTA":7960, "FPA":429, "TD":1807728 }

# archive actual sources (canonical, per Overall/ARCHIVE_SEGMENTS)
ARCHIVE_TBL = {
 "FTA":"hive_metastore.ariadm_arm_fta.stg_appeals_filtered",
 "UTA":"hive_metastore.ariadm_arm_uta.stg_appeals_filtered",
 "FPA":"hive_metastore.ariadm_arm_fpa.stg_filepreservedcases_filtered",
 "TD" :"hive_metastore.ariadm_arm_td.stg_td_filtered",   # NOTE: includes dept-519 cases (huge)
}
ACTIVE_TBL = "hive_metastore.ariadm_active_appeals.stg_segmentation_states"

_c=spark.read.option("multiline","true").json("dbfs:/configs/config.json")
env_name=_c.first()["env"].strip().lower(); lz_key=_c.first()["lz_key"].strip().lower()
KV=f"ingest{lz_key}-meta002-{env_name}"
cid=dbutils.secrets.get(KV,"SERVICE-PRINCIPLE-CLIENT-ID"); csec=dbutils.secrets.get(KV,"SERVICE-PRINCIPLE-CLIENT-SECRET"); tid=dbutils.secrets.get(KV,"SERVICE-PRINCIPLE-TENANT-ID")
for sa in [f"ingest{lz_key}curated{env_name}",f"ingest{lz_key}raw{env_name}"]:
    spark.conf.set(f"fs.azure.account.auth.type.{sa}.dfs.core.windows.net","OAuth")
    spark.conf.set(f"fs.azure.account.oauth.provider.type.{sa}.dfs.core.windows.net","org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
    spark.conf.set(f"fs.azure.account.oauth2.client.id.{sa}.dfs.core.windows.net",cid)
    spark.conf.set(f"fs.azure.account.oauth2.client.secret.{sa}.dfs.core.windows.net",csec)
    spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{sa}.dfs.core.windows.net",f"https://login.microsoftonline.com/{tid}/oauth2/token")
def col_ci(cols,name): return next((c for c in cols if c.lower()==name.lower()),None)

In [0]:
# ---- CELL 1 : gather ACTUALS (active per-state + archive per-segment) ----
A = spark.table(ACTIVE_TBL)
acc = col_ci(A.columns,"CaseNo"); atc = col_ci(A.columns,"TargetState")
active_counts = {r["TargetState"]: r["n"] for r in
                 A.select(trim(col(acc)).alias("CaseNo"), col(atc).alias("TargetState")).distinct()
                  .groupBy("TargetState").agg(count("*").alias("n")).collect()}
active_total_actual = A.select(trim(col(acc))).distinct().count()

archive_counts = {}
for seg, tbl in ARCHIVE_TBL.items():
    try:
        t = spark.table(tbl); cc = col_ci(t.columns,"CaseNo")
        archive_counts[seg] = t.select(trim(col(cc))).distinct().count()
    except Exception as e:
        archive_counts[seg] = None; log(f"  {seg} read ERR {str(e)[:80]}")

In [0]:
# ---- CELL 2 : build comparison rows ----
import pandas as pd
def _row(zone, name, expected, actual):
    if expected is None:
        status = "same-as-before (set previous count)"; diff = ""
    elif actual is None:
        status = "NO ACTUAL"; diff = ""
    else:
        diff = actual - expected; status = "MATCH" if diff == 0 else "DIFF"
    return {"zone":zone, "item":name, "expected":expected, "actual":actual, "diff":diff, "status":status}

rows=[]
for st in ACTIVE_EXPECTED: rows.append(_row("active", st, ACTIVE_EXPECTED[st], active_counts.get(st, 0)))
rows.append(_row("active", "TOTAL", pysum(ACTIVE_EXPECTED.values()), active_total_actual))
for seg in ARCHIVE_TBL: rows.append(_row("archive", seg, ARCHIVE_EXPECTED.get(seg), archive_counts.get(seg)))
df = pd.DataFrame(rows, columns=["zone","item","expected","actual","diff","status"])

n_diff = int((df["status"]=="DIFF").sum()); n_tbd = int(df["status"].str.startswith("same-as-before (set").sum())
log("="*74); log("EXPECTED (Bella) vs ACTUAL (pipeline)  -  active states + archive segments"); log("="*74)
log(f"verdict: {'ALL MATCH' if n_diff==0 else str(n_diff)+' DIFFERENCE(S)'}" + (f"   ({n_tbd} row(s) need a previous count set)" if n_tbd else ""))
log("")
try: log(df.to_string(index=False))
except Exception: log(str(rows))

In [0]:
# ---- CELL 3 : downloadable workbook + single print ----
buttons=""
try:
    try: import openpyxl  # noqa
    except Exception:
        import subprocess,sys; subprocess.run([sys.executable,"-m","pip","install","-q","openpyxl"])
    buf=io.BytesIO()
    with pd.ExcelWriter(buf, engine="openpyxl") as xw:
        df.to_excel(xw, sheet_name="expected_vs_actual", index=False)
    b64=base64.b64encode(buf.getvalue()).decode()
    buttons=(f'<div style="font-family:sans-serif"><a download="recon_expected_vs_actual.xlsx" '
      f'style="display:inline-block;background:#0b5cad;color:#fff;text-decoration:none;padding:8px 14px;border-radius:6px;font-size:13px" '
      f'href="data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{b64}">'
      f'&#11015; Download expected-vs-actual (Excel)</a></div>')
    displayHTML(buttons)
except Exception as e:
    log(f"(download build note: {str(e)[:120]})")

full="\n".join(REPORT)
try:
    user=spark.sql("SELECT current_user()").first()[0]; ts=datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    folder=f"/Workspace/Users/{user}/Results/recon_expected_vs_actual/{ts}"; dbutils.fs.mkdirs(f"file:{folder}")
    p=f"{folder}/recon_expected_vs_actual.txt"; open(p,"w").write(full); full+=f"\n\n>>> saved to: {p}"
except Exception as e: full+=f"\n(save note: {str(e)[:80]})"
print(full)